# 05 · Selección del modelo, prueba única y registro

1. **Champion y challenger**: las dos mejores familias por PR-AUC por evento en la CV (versiones optimizadas del
   notebook 04 y candidatos del 03), con la pregunta de si la diferencia supera la variación entre pliegues.
2. **Modelos finales**: se entrenan con **todo** el período de entrenamiento (2018-01 → 2019-09) con el mismo protocolo
   (todas las fallas + 15 % de los discos sanos, semilla 42).
3. **Prueba única en noviembre de 2019** (después de un mes de embargo), con todas sus filas: PR-AUC y F0,5 por
   evento, discos vistos y no vistos en entrenamiento, anticipación. **Esta es la única vez que se usa la prueba.**
4. **Métrica de negocio**: costo esperado por 1.000 SSD-mes con C_FP = 1 y C_FN = k, k ∈ [1, 50], frente a la regla
   de umbral; k de equilibrio. **Los umbrales τ se eligen con las predicciones out-of-fold de la CV**, nunca con la prueba.
5. **Tiempos y explicabilidad** (valores SHAP).
6. **Model Registry**: `ssd-failure-modelo` con alias `@champion` y `@challenger`; `docs/corridas.csv`.

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src import datos, pipeline as pl, experimentos as exp

pd.set_option("display.width", 220, "display.max_columns", 40, "display.float_format", "{:,.4f}".format)
sns.set_theme(style="whitegrid", context="notebook")
DATOS = Path(os.environ.get("DATOS_DIR", "/datos"))
FIG = Path("../docs/figuras"); FIG.mkdir(parents=True, exist_ok=True)
SEMILLA, FRAC = pl.SEMILLA, 0.15
T0 = time.time()

mlflow = pl.configurar_mlflow()
from mlflow import MlflowClient
cliente = MlflowClient()
manifiesto = datos.leer_manifiesto(DATOS / "obt_mc1")
META_DATOS = datos.metadatos_dataset(DATOS / "obt_mc1")
eda = json.loads(Path("../docs/eda_decisiones.json").read_text())
print("commit:", pl.commit_git(), "· experimento:", pl.NOMBRE_EXPERIMENTO)

## 1. Champion y challenger según la CV

In [ ]:
r03 = exp.resultados(DATOS / "oof" / "03")
r04 = exp.resultados(DATOS / "oof" / "04")
filas = []
for nombre, r in {**r03, **r04}.items():
    if r["fase"] == "baseline" or "sin_degradacion" in nombre:
        continue
    filas.append({"nombre": nombre, "familia": r["familia"], "fase": r["fase"], "run_id": r["run_id"],
                  "cv_media": r["cv_pr_auc_evento_media"], "cv_std": r["cv_pr_auc_evento_std"],
                  "train": r["train_pr_auc_evento"], "oof": str((DATOS / "oof" / ("04" if nombre in r04 else "03")) / f"{nombre}.parquet")})
ranking = pd.DataFrame(filas).sort_values("cv_media", ascending=False).set_index("nombre")
# Una sola versión por familia: la mejor (optimizada si el tuning mejoró)
por_familia = ranking.groupby("familia", sort=False).head(1)
CHAMPION, CHALLENGER = por_familia.index[0], por_familia.index[1]
c, d = por_familia.loc[CHAMPION], por_familia.loc[CHALLENGER]
diferencia = c["cv_media"] - d["cv_media"]
print(f"Champion: {CHAMPION} (CV {c['cv_media']:.4f} ± {c['cv_std']:.4f})")
print(f"Challenger: {CHALLENGER} (CV {d['cv_media']:.4f} ± {d['cv_std']:.4f})")
print(f"Diferencia {diferencia:.4f} {'>' if diferencia > max(c['cv_std'], d['cv_std']) else '≤'} "
      f"la std entre pliegues ({max(c['cv_std'], d['cv_std']):.4f}): "
      + ("diferencia clara" if diferencia > max(c['cv_std'], d['cv_std']) else "NO es una diferencia clara; la prueba y el costo deciden con más información"))
ranking

In [ ]:
def params_de(nombre):
    # Hiperparámetros con que se entrenó la corrida (optimizada: los del mejor trial; candidato: por defecto)
    tuning = json.loads(Path("../docs/tuning_resultado.json").read_text()) if Path("../docs/tuning_resultado.json").exists() else {}
    familia = ranking.loc[nombre, "familia"]
    if nombre.endswith("_optimizado") and familia in tuning:
        import optuna
        from src import tuning as tn
        return tn.ESPACIOS[familia](optuna.trial.FixedTrial(tuning[familia]["mejores_hiperparametros"]))
    return None

ELEGIDOS = {"champion": CHAMPION, "challenger": CHALLENGER}
PARAMS = {rol: params_de(nombre) for rol, nombre in ELEGIDOS.items()}
PARAMS

## 2. Umbrales de operación elegidos con la CV (no con la prueba)

Con las predicciones *out-of-fold* de la CV se construye la curva precisión-recall por evento de cada modelo y de la
regla. De ahí salen (a) el τ de **operación** que maximiza F0,5 por evento y (b) para cada k ∈ [1, 50], el τ que
minimiza el costo por 1.000 SSD-mes. Estos τ se aplican tal cual en la prueba.

In [ ]:
def curva_oof(ruta):
    o = pd.read_parquet(ruta)
    curva = pl.curva_evento(o["SSD_KEY"], o["OBSERVATION_DATE"], o["DAYS_TO_FAILURE"], o["score"].to_numpy())
    return o, curva, pl.ssd_meses(o["SSD_KEY"], o["OBSERVATION_DATE"])

def tau_f05(curva, beta=0.5):
    p, r = curva.precision, curva.recall
    f = np.where(p + r > 0, (1 + beta**2) * p * r / np.maximum(beta**2 * p + r, 1e-12), 0)
    i = int(np.argmax(f))
    return float(curva.umbrales[i]), float(f[i])

regla_nombre = "regla_" + eda["atributo_regla"]
OOF = {rol: ranking.loc[n, "oof"] for rol, n in ELEGIDOS.items()} | {"regla": str(DATOS / "oof" / "03" / f"{regla_nombre}.parquet")}
UMBRALES, TAUS_K = {}, {}
for rol, ruta in OOF.items():
    o, curva, exposicion = curva_oof(ruta)
    UMBRALES[rol] = tau_f05(curva)
    TAUS_K[rol] = pl.barrer_k(curva, exposicion)
    print(f"{rol:10s}: PR-AUC OOF {curva.pr_auc:.4f} · τ F0,5 = {UMBRALES[rol][0]:.5f} (F0,5 OOF {UMBRALES[rol][1]:.4f})")

## 3. Modelos finales (todo el período de entrenamiento)

In [ ]:
train = datos.cargar_cache(DATOS / "cache_R", hasta=datos.FIN_ENTRENAMIENTO, sha_esperado=manifiesto["dataset_sha256"])
prueba = datos.cargar_cache(DATOS / "cache_R", desde=datos.INICIO_PRUEBA, hasta=datos.FIN_PRUEBA)
assert (prueba["OBSERVATION_DATE"].min() - train["OBSERVATION_DATE"].max()).days > 30, "embargo"
COLUMNAS = pl.columnas_modelo(train, eda["variante"])
meta_tr = train[["SSD_KEY", "OBSERVATION_DATE", "DAYS_TO_FAILURE", "TARGET_30D"]]
pos = pl.muestra_por_disco(meta_tr, FRAC, SEMILLA)
idx = train.columns.get_indexer(COLUMNAS)
X_tr, y_tr = train.iloc[pos, idx], meta_tr["TARGET_30D"].to_numpy()[pos]
X_te = prueba[COLUMNAS]
print(f"Entrenamiento: {len(pos):,} filas ({FRAC:.0%} de los sanos + todas las fallas) · "
      f"prueba: {len(prueba):,} filas, {prueba['OBSERVATION_DATE'].min().date()} → {prueba['OBSERVATION_DATE'].max().date()}")

MODELOS, TIEMPOS = {}, {}
for rol, nombre in ELEGIDOS.items():
    familia = ranking.loc[nombre, "familia"]
    m = pl.construir_modelo(familia, PARAMS[rol], SEMILLA)
    t = time.perf_counter(); m.fit(X_tr, y_tr); TIEMPOS[rol] = {"entrenamiento_s": time.perf_counter() - t}
    MODELOS[rol] = m
regla = pl.construir_modelo("regla", {"atributo": eda["atributo_regla"]}, SEMILLA)
t = time.perf_counter(); regla.fit(X_tr, y_tr); TIEMPOS["regla"] = {"entrenamiento_s": time.perf_counter() - t}
MODELOS["regla"] = regla
pd.DataFrame(TIEMPOS).T

## 4. Prueba única: noviembre de 2019

In [ ]:
discos_train = set(train["SSD_KEY"].cat.categories[np.unique(train["SSD_KEY"].cat.codes)])
visto = prueba["SSD_KEY"].astype(str).isin(discos_train).to_numpy()
SCORES, filas = {}, []
for rol, m in MODELOS.items():
    t = time.perf_counter(); s = m.predict_proba(X_te)[:, 1]; TIEMPOS[rol]["inferencia_s_por_millon"] = (time.perf_counter() - t) / len(X_te) * 1e6
    SCORES[rol] = s
    tau = UMBRALES[rol][0]
    for grupo, mask in (("todos", np.ones(len(s), bool)), ("discos vistos", visto), ("discos no vistos", ~visto)):
        sub = prueba.loc[mask]
        curva = pl.curva_evento(sub["SSD_KEY"], sub["OBSERVATION_DATE"], sub["DAYS_TO_FAILURE"], s[mask])
        op = pl.metricas_evento(sub["SSD_KEY"], sub["OBSERVATION_DATE"], sub["DAYS_TO_FAILURE"], s[mask], tau)
        filas.append({"modelo": rol, "grupo": grupo, "filas": int(mask.sum()), "eventos": op["eventos"],
                      "pr_auc_evento": curva.pr_auc, "tau (CV)": tau, "precision_evento": op["precision_evento"],
                      "recall_evento": op["recall_evento"], "f05_evento": op["f05_evento"],
                      "alertas": op["alertas"], "aciertos": op["aciertos"], "alertas_tardias": op["alertas_tardias"]})
PRUEBA = pd.DataFrame(filas).set_index(["modelo", "grupo"])
PRUEBA

In [ ]:
# Anticipación: días entre la primera alerta (score ≥ τ) y la falla, en los discos que fallan
filas = []
for rol in MODELOS:
    a = prueba.loc[SCORES[rol] >= UMBRALES[rol][0], ["SSD_KEY", "OBSERVATION_DATE", "DAYS_TO_FAILURE"]]
    primera = a.sort_values("OBSERVATION_DATE").groupby("SSD_KEY", observed=True).head(1).dropna(subset=["DAYS_TO_FAILURE"])
    filas.append({"modelo": rol, "discos_que_fallan_alertados": len(primera),
                  "anticipacion_mediana_dias": primera["DAYS_TO_FAILURE"].median(),
                  "en_ventana_7_30": int(primera["DAYS_TO_FAILURE"].between(7, 30).sum()),
                  "tardias_1_6": int(primera["DAYS_TO_FAILURE"].between(1, 6).sum()),
                  "tempranas_>30": int((primera["DAYS_TO_FAILURE"] > 30).sum())})
pd.DataFrame(filas).set_index("modelo")

## 5. Métrica de negocio: costo por 1.000 SSD-mes

In [ ]:
exposicion = pl.ssd_meses(prueba["SSD_KEY"], prueba["OBSERVATION_DATE"])
COSTOS = {rol: pl.costo_con_tau(prueba["SSD_KEY"], prueba["OBSERVATION_DATE"], prueba["DAYS_TO_FAILURE"],
                                SCORES[rol], TAUS_K[rol]) for rol in MODELOS}
fig, ax = plt.subplots(figsize=(10, 4.5))
for rol, c in COSTOS.items():
    ax.plot(c["k"], c["costo"], label=f"{rol} ({ELEGIDOS.get(rol, regla_nombre)})", lw=2)
ax.plot(COSTOS["regla"]["k"], COSTOS["regla"]["costo_sin_modelo"], "k:", label="sin modelo (no alertar nunca)")
ax.set_xlabel("k = costo de una falla no anticipada / costo de una falsa alarma")
ax.set_ylabel("costo por 1.000 SSD-mes (prueba)")
ax.set_title("Costo esperado en la prueba con τ elegido en la CV")
ax.legend(); fig.tight_layout(); fig.savefig(FIG / "05_costo_vs_k.png", dpi=150); plt.show()

k_eq = {rol: pl.k_equilibrio(COSTOS[rol], COSTOS["regla"]) for rol in ELEGIDOS}
print("k de equilibrio frente a la regla (menor k desde el que el modelo cuesta menos):", k_eq)
tabla_k = pd.concat({rol: c.set_index("k")[["costo", "alertas", "aciertos"]] for rol, c in COSTOS.items()}, axis=1)
tabla_k.loc[[1, 5, 10, 20, 30, 50]]

## 6. Explicabilidad (valores SHAP)

In [ ]:
# XGBoost calcula valores SHAP exactos de sus árboles (pred_contribs) en GPU; se explica el primer modelo de árboles
import xgboost as xgb
rol_arbol = next((r for r in ("champion", "challenger") if ranking.loc[ELEGIDOS[r], "familia"] in {"xgboost", "bosque"}), None)
if rol_arbol is None:
    print("Ni el champion ni el challenger son de árboles: no se calculan valores SHAP exactos")
else:
    m = MODELOS[rol_arbol]
    nombres = list(COLUMNAS) + [pl.FEATURE_DEGRADACION]
    rng = np.random.default_rng(SEMILLA)
    pos_pos = np.flatnonzero(prueba["TARGET_30D"].to_numpy() == 1)
    muestra = np.concatenate([rng.choice(pos_pos, min(1000, len(pos_pos)), replace=False),
                              rng.choice(len(prueba), 4000, replace=False)])
    Xm = m.named_steps["prep"].transform(X_te.iloc[muestra])
    contrib = m.named_steps["modelo"].get_booster().predict(xgb.DMatrix(Xm), pred_contribs=True)[:, :-1]
    imp = pd.Series(np.abs(contrib).mean(0), index=nombres).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 6))
    imp.head(20)[::-1].plot.barh(ax=ax, color="#4C72B0")
    ax.set_xlabel("|SHAP| medio (log-odds)"); ax.set_title(f"Features más influyentes · {rol_arbol} ({ELEGIDOS[rol_arbol]})")
    fig.tight_layout(); fig.savefig(FIG / "05_shap_importancia.png", dpi=150); plt.show()
    # Un ejemplo: una alerta acertada (disco que falla 7–30 días después, con score ≥ τ)
    s, tau = SCORES[rol_arbol], UMBRALES[rol_arbol][0]
    acierto = np.flatnonzero((s >= tau) & prueba["DAYS_TO_FAILURE"].between(7, 30).to_numpy())
    if len(acierto):
        i = acierto[np.argmax(s[acierto])]
        ci = m.named_steps["modelo"].get_booster().predict(
            xgb.DMatrix(m.named_steps["prep"].transform(X_te.iloc[[i]])), pred_contribs=True)[0, :-1]
        ej = pd.Series(ci, index=nombres).sort_values(key=np.abs, ascending=False).head(10)
        print(f"Alerta explicada: disco con score {s[i]:.4f} (τ {tau:.4f}), falla {prueba['DAYS_TO_FAILURE'].iloc[i]:.0f} días después")
        display(pd.DataFrame({"contribucion_log_odds": ej, "valor": X_te.iloc[i].reindex(ej.index)}))
    imp.head(15)

## 7. Registro: corridas finales, Model Registry y `corridas.csv`

In [ ]:
VERSIONES = {}
for rol, nombre in ELEGIDOS.items():
    familia = ranking.loc[nombre, "familia"]
    fila = PRUEBA.loc[(rol, "todos")]
    metricas = {"test_pr_auc_evento": fila["pr_auc_evento"], "test_f05_evento": fila["f05_evento"],
                "test_precision_evento": fila["precision_evento"], "test_recall_evento": fila["recall_evento"],
                "test_pr_auc_evento_discos_no_vistos": PRUEBA.loc[(rol, "discos no vistos"), "pr_auc_evento"],
                "tau_operacion": UMBRALES[rol][0], "cv_pr_auc_evento_media": ranking.loc[nombre, "cv_media"],
                "cv_pr_auc_evento_std": ranking.loc[nombre, "cv_std"],
                "segundos_entrenamiento": TIEMPOS[rol]["entrenamiento_s"],
                "inferencia_s_por_millon": TIEMPOS[rol]["inferencia_s_por_millon"],
                "k_equilibrio_vs_regla": k_eq[rol] if k_eq[rol] is not None else np.nan}
    run_id = pl.registrar_corrida(
        nombre=f"final_{rol}_{familia}", fase="final", familia=familia, modelo=MODELOS[rol], seed=SEMILLA,
        metricas=metricas, datos=META_DATOS, X_ejemplo=X_te.head(5),
        params_extra={"frac_sanos_train": FRAC, "entrenamiento": "2018-01-03 a 2019-09-30",
                      "prueba": "2019-11-01 a 2019-11-30", "origen": nombre,
                      **{f"hp_{k}": v for k, v in (PARAMS[rol] or {}).items()}},
        tags_extra={"rol": rol, "corrida_cv": ranking.loc[nombre, "run_id"]})
    version = mlflow.register_model(f"runs:/{run_id}/pipeline", pl.NOMBRE_MODELO_REGISTRO)
    cliente.set_registered_model_alias(pl.NOMBRE_MODELO_REGISTRO, rol, version.version)
    VERSIONES[rol] = {"run_id": run_id, "version": version.version, "test_pr_auc_evento": fila["pr_auc_evento"]}
Path("../docs/registro.json").write_text(json.dumps({"modelo": pl.NOMBRE_MODELO_REGISTRO, **VERSIONES,
                                                     "dataset_sha256": manifiesto["dataset_sha256"]}, indent=2))
VERSIONES

In [ ]:
corridas = pl.exportar_corridas("../docs/corridas.csv")
print(f"docs/corridas.csv: {len(corridas)} corridas")
print(corridas["tags.fase"].value_counts().to_string())
print(f"Tiempo total del notebook: {(time.time()-T0)/60:.1f} min")